In [ ]:
%pip -q install "pydantic>=2,<3" requests pandas

In [ ]:
import json
import os
import time
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator


def locate_input(*names):
    for name in names:
        path = Path(name)
        if path.is_file():
            return path
    raise FileNotFoundError("Upload the input file. Checked: " + ", ".join(names))


notes_path = locate_input(
    "data/followup_notes.csv", "followup_notes.csv",
    "data/ecWvPsdV2FNmRZAa.csv", "ecWvPsdV2FNmRZAa.csv",
)
first_ten = pd.read_csv(
    notes_path, dtype={"note_id": "string", "appointment_id": "string"}
).head(10).copy()
if len(first_ten) != 10 or first_ten["note_id"].isna().any() or not first_ten["note_id"].is_unique:
    raise ValueError("The first ten rows must have ten distinct, nonmissing note IDs.")
note_texts = first_ten.set_index("note_id")["note_text"]
missing_ids = {"N001", "N007", "N009"}.difference(note_texts.index)
if missing_ids:
    raise ValueError(f"Expected notes missing from the first ten: {sorted(missing_ids)}")
print(f"Loaded {len(first_ten)} notes from {notes_path}.")

In [ ]:
MODEL_NAME = "@cf/meta/llama-3.3-70b-instruct-fp8-fast"
ACCOUNT_ID = (os.getenv("CLOUDFLARE_ACCOUNT_ID") or input("Cloudflare Account ID: ")).strip()
API_TOKEN = (os.getenv("CLOUDFLARE_API_TOKEN") or getpass("Cloudflare Workers AI token: ")).strip()
if not ACCOUNT_ID or not API_TOKEN or any(c.isspace() for c in API_TOKEN):
    raise ValueError("Enter the Account ID and the token value from Workers AI → Use REST API.")
RUN_URL = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/run/{MODEL_NAME}"
CALL_INTERVAL_SECONDS = 13
# Keep the session counter if this configuration cell is run again.
api_call_count = globals().get("api_call_count", 0)
next_request_at = globals().get("next_request_at", 0.0)

In [ ]:
ALLOWED_CAUSES = (
    "forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
    "illness_or_emergency", "work_or_schedule", "other",
)
FIELDS = ("root_cause", "confidence", "summary", "recommended_action")
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": list(FIELDS),
}

# The common system instruction leaves the label list and ambiguity rule to V2.
SYSTEM_TEXT = (
    "You are a CareLoop clinic operations analyst reviewing missed-appointment notes. "
    "Base your response on the supplied note. Use operational language."
)


def build_craft_prompt(note_text):
    return (
        "Context: A front-desk employee recorded this note after a missed appointment.\n"
        "Role: You are a healthcare operations analyst.\n"
        f"Action: Choose one root cause from {', '.join(ALLOWED_CAUSES)}. "
        "Use other when the reason is unclear. Do not invent a reason.\n"
        "Tone: Concise, objective and practical.\n"
        "Format: Return JSON with root_cause, confidence (0 to 1), summary "
        "(at most 25 words), and recommended_action (a front-desk action).\n\n"
        f"Note:\n{note_text}"
    )


class FollowupInsight(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator("root_cause")
    @classmethod
    def allowed_cause(cls, value):
        if value not in ALLOWED_CAUSES:
            raise ValueError("root_cause is outside the allowed list")
        return value

    @field_validator("summary")
    @classmethod
    def short_summary(cls, value):
        if not 1 <= len(value.split()) <= 25:
            raise ValueError("summary must contain 1 to 25 words")
        return value

In [ ]:
def json_member_source(object_text, member):
    """Read one member of an already parsed JSON object, keeping its original text."""
    decoder = json.JSONDecoder()
    remaining = object_text.lstrip()[1:].lstrip()  # after the opening brace
    while remaining and not remaining.startswith("}"):
        name, key_end = decoder.raw_decode(remaining)
        value_text = remaining[key_end:].lstrip()[1:].lstrip()  # after the colon
        _, value_end = decoder.raw_decode(value_text)
        if name == member:
            return value_text[:value_end]
        remaining = value_text[value_end:].lstrip()
        if not remaining.startswith(","):
            break
        remaining = remaining[1:].lstrip()
    raise ValueError(f"Response did not contain JSON member {member!r}.")


def call_cloudflare(prompt_text):
    global api_call_count, next_request_at
    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM_TEXT},
            {"role": "user", "content": prompt_text},
        ],
        "temperature": 0,
        "max_tokens": 1000,
        "stream": False,
        "response_format": {"type": "json_schema", "json_schema": SCHEMA},
    }
    time.sleep(max(0.0, next_request_at - time.monotonic()))
    api_call_count += 1
    try:
        reply = requests.post(
            RUN_URL,
            headers={"Authorization": f"Bearer {API_TOKEN}"},
            json=payload,
            timeout=(10, 90),
        )
    finally:
        next_request_at = time.monotonic() + CALL_INTERVAL_SECONDS

    try:
        envelope = reply.json()
    except ValueError:
        raise RuntimeError(f"Cloudflare HTTP {reply.status_code} returned a non-JSON body.") from None
    if not isinstance(envelope, dict):
        raise RuntimeError(f"Cloudflare HTTP {reply.status_code} returned an unexpected response.")
    if not reply.ok or envelope.get("success") is False:
        errors = envelope.get("errors") or []
        details = " | ".join(
            str(item.get("message") or item.get("code")) for item in errors
            if isinstance(item, dict)
        ) or reply.reason
        raise RuntimeError(f"Cloudflare HTTP {reply.status_code}: {details}")

    result = envelope.get("result")
    if not isinstance(result, dict) or "response" not in result:
        raise RuntimeError("Cloudflare returned no result.response for this request.")
    if isinstance(result["response"], str):
        return result["response"]
    result_source = json_member_source(reply.text, "result")
    return json_member_source(result_source, "response")

In [ ]:
def analyse_note(note_text, *, minimal=False, audit=None):
    """Return a validated four-field dict. If supplied, audit retains the original reply."""
    prompt = f"Classify this note:\n{note_text}" if minimal else build_craft_prompt(note_text)
    raw_text = call_cloudflare(prompt)
    if audit is not None:
        audit["raw_response"] = raw_text  # retain even if json.loads fails
    data = json.loads(raw_text)
    if not isinstance(data, dict):
        raise ValueError("The model reply must be a JSON object.")
    if audit is not None:
        audit.update({name: data[name] for name in FIELDS if name in data})
    FollowupInsight(**data)
    return data


def review_note(note_id, note_text, *, minimal=False):
    row = {name: "" for name in FIELDS}
    row.update(validation_status="invalid", raw_response="")
    if pd.isna(note_text) or str(note_text).strip() == "":
        row.update(
            root_cause="other", confidence=0.0, summary="No note recorded",
            recommended_action="Re-call patient", validation_status="skipped_blank",
        )
        return row
    try:
        analyse_note(note_text, minimal=minimal, audit=row)
    except (ValueError, RuntimeError, requests.RequestException) as exc:
        print(f"{note_id} | {'V1' if minimal else 'CRAFT'} | {exc}")
    else:
        row["validation_status"] = "valid"
    return row

In [ ]:
pilot_audit = {}
try:
    n001_data = analyse_note(note_texts.at["N001"], audit=pilot_audit)
except (ValueError, RuntimeError, requests.RequestException):
    if pilot_audit.get("raw_response"):
        print("Model reply for N001:", repr(pilot_audit["raw_response"]))
    raise
print("Task 4.1 (N001):", n001_data)

In [ ]:
records = []
for note in first_ten.itertuples(index=False):
    reviewed = review_note(note.note_id, note.note_text)
    records.append({"note_id": note.note_id, "appointment_id": note.appointment_id, **reviewed})

columns = ["note_id", "appointment_id", *FIELDS, "validation_status", "raw_response"]
analysed_df = pd.DataFrame(records, columns=columns)
output_path = Path("notes_analysed.csv")
analysed_df.to_csv(output_path, index=False)
print(f"Saved {len(analysed_df)} rows to {output_path.resolve()}")
print(analysed_df[["note_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))

In [ ]:
bad_cases = [
    {"root_cause": "other", "confidence": 1.7, "summary": "Valid summary",
     "recommended_action": "Call patient"},
    {"root_cause": "other", "confidence": 0.8, "summary": "Missing action field"},
]
for index, bad in enumerate(bad_cases, 1):
    try:
        FollowupInsight(**bad)
    except ValidationError as exc:
        print(f"Bad case {index}:\n{exc}\n")

In [ ]:
batch_by_note = analysed_df.set_index("note_id")
compare_rows = []
for note_id in ("N007", "N009"):
    v1 = review_note(note_id, note_texts.at[note_id], minimal=True)
    v2 = batch_by_note.loc[note_id]
    for label, record in (("V1 (minimal)", v1), ("V2 (CRAFT)", v2)):
        cause = record["root_cause"]
        compare_rows.append({
            "note_id": note_id, "prompt": label, "root_cause": cause,
            "in_allowed": isinstance(cause, str) and cause in ALLOWED_CAUSES,
            "validation_status": record["validation_status"],
        })

comparison = pd.DataFrame(compare_rows)
print(comparison.to_string(index=False))
print(f"API requests attempted in this runtime: {api_call_count}")

In [ ]:
clean_path = locate_input("appointments_clean.csv", "data/appointments_clean.csv")
clean_df = pd.read_csv(
    clean_path, usecols=["appointment_id", "reminder_sent"],
    dtype={"appointment_id": "string", "reminder_sent": "string"},
)
merged = analysed_df.merge(
    clean_df, on="appointment_id", how="left", validate="many_to_one", indicator=True
)
unmatched = merged.loc[merged["_merge"] != "both", "appointment_id"].tolist()
if unmatched:
    raise ValueError(f"These note appointment IDs are missing from the clean table: {unmatched}")
if not merged["reminder_sent"].isin(["Yes", "No"]).all():
    raise ValueError("Expected Yes or No for every joined reminder_sent value.")

# Use a display label for missing or non-text causes without changing the saved response.
display_causes = merged["root_cause"].map(
    lambda value: value if isinstance(value, str) and value.strip() else "(missing or non-text cause)"
).rename("root_cause")
cross_tab = pd.crosstab(display_causes, merged["reminder_sent"])
cross_tab = cross_tab.reindex(columns=["No", "Yes"], fill_value=0)
print("Merged rows:", len(merged))
print(cross_tab.to_string())
print("Notes by reminder status:", cross_tab.sum().to_dict())